# Week 6 participant practical: diagram summaries, distances and validation

This is the student investigation, built around a noisy circle, a small perturbation and a coordinate-shuffled surrogate. The setup and mathematical framing are supplied. You must record predictions, complete short computational steps, check intermediate objects and justify an interpretation.

Diagrams are not conclusions. This laboratory compares direct diagram distances, Betti curves, vector summaries and a surrogate null. Every summary forgets something. All examples use $H_1$ over $\mathbb F_2$.

**Working rule.** Run one section at a time. Before each TODO, state what shape, dimension or direction you expect in the output. Optional extensions come only after the core checkpoints agree.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
RNG=np.random.default_rng(3024)
from ripser import ripser
from persim import bottleneck, wasserstein

def betti_curve(D,grid):
    return np.array([np.sum((D[:,0]<=a)&(a<D[:,1])) for a in grid])

def persistence_image(D,bounds=(0,2,0,2),n=24,sigma=.09):
    finite=D[np.isfinite(D[:,1])]
    xs=np.linspace(bounds[0],bounds[1],n); ys=np.linspace(bounds[2],bounds[3],n)
    X,Y=np.meshgrid(xs,ys); img=np.zeros_like(X)
    for b,d in finite:
        weight=d-b; img+=weight*np.exp(-((X-b)**2+(Y-d)**2)/(2*sigma**2))
    return img

def max_persistence(D):
    F=D[np.isfinite(D[:,1])]; return float(np.max(F[:,1]-F[:,0])) if len(F) else 0.

## Hand checkpoint: why the diagonal is available

Let $D=\{(0,2),(1,1.4)\}$ and $E=\{(0.1,2.1)\}$. Match the long points and send the short point to the diagonal.

Predict the $L_\infty$ cost of each match. The distance from $(b,d)$ to the diagonal is $(d-b)/2$.

In [ ]:
long_match_cost = max(abs(0.0 - 0.1), abs(2.0 - 2.1))
short_to_diagonal = (1.4 - 1.0) / 2
candidate_bottleneck = max(long_match_cost, short_to_diagonal)
print('long match:', long_match_cost)
print('short point to diagonal:', short_to_diagonal)
print('candidate bottleneck cost:', candidate_bottleneck)
assert np.isclose(candidate_bottleneck, 0.2)

## 1. Observe: participant checkpoint

Create a noisy circle, a slightly perturbed copy, and a coordinate-shuffled surrogate that preserves the separate $x$ and $y$ marginals but destroys their pairing.

In [ ]:
n=80; theta=np.linspace(0,2*np.pi,n,endpoint=False)
base=np.c_[np.cos(theta),np.sin(theta)]+.05*RNG.normal(size=(n,2))
perturbed=base+.025*RNG.normal(size=base.shape)
surrogate=base.copy(); surrogate[:,1]=RNG.permutation(surrogate[:,1])
datasets={'base':base,'perturbed':perturbed,'surrogate':surrogate}
diagrams={k:ripser(P,maxdim=1)['dgms'][1] for k,P in datasets.items()}
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,(name,P) in zip(axes,datasets.items()): ax.scatter(*P.T,s=12); ax.set_title(name); ax.set_aspect('equal')
plt.show()

## 2. Predict: participant checkpoint

1. Which pair should have the smallest bottleneck distance?
2. When might Wasserstein distance react more strongly than bottleneck distance?
3. Can two diagrams share maximum persistence but differ substantially elsewhere?
4. What property does the shuffled surrogate preserve, and what structure does it destroy?

## 3. Implement: participant checkpoint

### A. Direct distances

Bottleneck records the worst matched cost; Wasserstein accumulates matching costs. Both allow diagonal matches.

**Checkpoint.** First print the number of finite $H_1$ intervals in each diagram. Then fill the two function calls. Confirm that every distance from a diagram to itself is zero.

In [ ]:
for name, diagram in diagrams.items():
    print(name, 'finite H1 intervals:', np.sum(np.isfinite(diagram[:, 1])))

distance_rows = []
for other in ['perturbed', 'surrogate']:
    # TODO: replace None with the appropriate distance calls.
    bottleneck_value = None
    wasserstein_value = None
    distance_rows.append((other, bottleneck_value, wasserstein_value))
print(distance_rows)

### B. Betti curves and persistence images

A Betti curve counts intervals alive at each scale. A persistence image smooths weighted birth-death points onto a fixed grid.

**Checkpoint.** Evaluate `betti_curve(diagrams['base'], grid)` at five grid locations before plotting all 150. For the image, check its array shape and whether all entries are non-negative.

In [ ]:
grid = np.linspace(0, 2, 150)
base_curve = betti_curve(diagrams['base'], grid)
print('five base-curve checks:', base_curve[::30])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
# TODO: plot one Betti curve per diagram on axes[0], including labels.
# TODO: calculate the base persistence image and display it on axes[1].
axes[0].set(xlabel='distance threshold', ylabel='beta_1')
plt.show()

## 4. Compare: participant checkpoint

Generate coordinate-shuffled surrogates through the **complete** pipeline. Begin with five repetitions to check the loop, then use 40.

Before running, state what the shuffle preserves and destroys. Choose `maximum persistence` as the statistic before seeing the null distribution. This is a teaching randomisation pattern, not automatically a valid scientific test.

In [ ]:
n_surrogates = 5  # checkpoint; change to 40 after the loop works
null_values = []
for repetition in range(n_surrogates):
    shuffled = base.copy()
    # TODO: independently permute the y coordinate.
    # TODO: compute its H1 diagram and append maximum persistence.
    pass

observed = max_persistence(diagrams['base'])
print('observed:', observed, 'null count:', len(null_values))
# TODO: after setting n_surrogates=40, plot the null histogram and observed line.

## 5. Interpret: participant checkpoint

1. Which summary emphasises one worst feature, and which accumulates many differences?
2. What information is lost by a Betti curve or persistence image?
3. What exchangeability or mechanism would justify the surrogate?
4. Why is an empirical exceedance not automatically a scientific p-value?
5. Which non-topological baseline should be evaluated beside the diagram summary?

**◇ Object check.** Diagram, distance, curve, image and scalar test statistic are successive summaries. None reconstructs the observed system.